# 検証：Fine-tuning による自律的なルーティング学習

## 概要
このノートブックでは、`VectorDBSynapse` を追加した後、少量のデータセットを用いてモデル（ルーターとエンコーダ）の追加学習（Fine-tuning）を行います。
これにより、特定の入力特徴量（質問など）に対して、ルーターが自律的に `VectorDBSynapse` を選択し、トラフィックを流すように適応していくプロセスを検証します。


In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ローカルパスの追加
sys.path.append(os.path.abspath('..'))

from src.sra_reference import SRAModel, VectorDBSynapse
from src.constants import PAD, BOS, EOS

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# 1. モデルの初期化
VOCAB_SIZE = 128
DIM = 64
LAYERS = 2
NUM_SYNAPSES = 4
K = 2
SYN_HIDDEN = 128

model = SRAModel(
    vocab_size=VOCAB_SIZE, 
    dim=DIM, 
    layers=LAYERS, 
    num_synapses=NUM_SYNAPSES, 
    k=K, 
    syn_hidden=SYN_HIDDEN
).to(device)

print(f"初期のシナプス数: {model.blocks[0].router.num_synapses}")


In [ ]:
# 2. VectorDBSynapse の追加とダミーデータの登録
def vectordb_factory():
    db = VectorDBSynapse(dim=DIM)
    # ダミーの知識を登録 (例えば、"What is SRA?" -> "SRA is a modular architecture.")
    # 今回は単純化のため、ランダムなキーバリューを登録します。
    # 実際の運用では、テキストをエンコードしてベクトル化したものを登録します。
    key1 = torch.randn(DIM)
    val1 = torch.randn(DIM)
    db.add_knowledge(key1, val1)
    
    key2 = torch.randn(DIM)
    val2 = torch.randn(DIM)
    db.add_knowledge(key2, val2)
    return db

def emb_factory():
    # VectorDBSynapse 用の初期 Router Embedding
    return torch.randn(DIM)

model.add_custom_synapse(vectordb_factory, emb_factory)
model = model.to(device)

vdb_synapse_idx = NUM_SYNAPSES
print(f"追加後のシナプス数: {model.blocks[0].router.num_synapses}")
print(f"VectorDBSynapse のインデックス: {vdb_synapse_idx}")


In [ ]:
# 3. ファインチューニング用データセットの作成
BATCH_SIZE = 16
SEQ_LEN = 12

# VectorDBSynapse に答えさせたい質問（タスク）のダミーデータ
# 入力 (x) と期待される出力 (y) を生成
def make_vdb_batch():
    x = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)
    y = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)
    y_in = torch.cat([torch.full((BATCH_SIZE, 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    return x, y_in, y

# 学習前のルーティング確率を確認するための固定バッチ
test_x, test_y_in, test_y = make_vdb_batch()


In [ ]:
# 4. 学習前のルーティング確率の確認
def get_vdb_routing_prob(model, x, y_in):
    model.eval()
    with torch.no_grad():
        _, router_logits, _ = model(x, y_in)
        last_block_logits = router_logits[-1][:, x.size(1):, :]
        weights = torch.softmax(last_block_logits, dim=-1)
        # VectorDBSynapse の確率の平均を取得
        vdb_prob = weights[:, :, vdb_synapse_idx].mean().item()
    return vdb_prob

prob_before = get_vdb_routing_prob(model, test_x, test_y_in)
print(f"学習前の VectorDBSynapse の選択確率: {prob_before:.4f}")


In [ ]:
# 5. ファインチューニングの実行
# 目標: VectorDBSynapse が選ばれるようにする
# 今回のデモでは、Loss計算に直接 VectorDBSynapse のルーティング重みへの報酬を組み込むか、
# Target出力とVectorDBの出力が一致するように最適化します。
# 最も直接的な方法は、教師ありルーティング（VectorDBのルーターロジットを最大化するLoss）です。

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 100

history_probs = []
history_loss = []

model.train()
for epoch in range(EPOCHS):
    x, y_in, y = make_vdb_batch()
    optimizer.zero_grad()
    
    outputs, router_logits, _ = model(x, y_in)
    
    # 標準の CrossEntropy Loss (ここではダミー出力を学習)
    ce_loss = F.cross_entropy(outputs.reshape(-1, VOCAB_SIZE), y.reshape(-1), ignore_index=PAD)
    
    # ルーティングへの誘導 Loss (VectorDBSynapse へのルーティング確率を上げる)
    # すべての層の target positions において、vdb_synapse_idx のロジットを最大化
    routing_loss = 0
    for logits in router_logits:
        tgt_logits = logits[:, x.size(1):, :]
        # vdb_synapse_idx 以外に対する CrossEntropy と見なすこともできる
        # ここでは単純に -log(softmax) を最小化
        probs = torch.softmax(tgt_logits, dim=-1)
        vdb_probs = probs[:, :, vdb_synapse_idx]
        routing_loss += -torch.log(vdb_probs + 1e-8).mean()
        
    total_loss = ce_loss + 0.1 * routing_loss
    
    total_loss.backward()
    optimizer.step()
    
    # 記録用
    if (epoch + 1) % 10 == 0:
        current_prob = get_vdb_routing_prob(model, test_x, test_y_in)
        history_probs.append(current_prob)
        history_loss.append(total_loss.item())
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss.item():.4f} - VectorDB Prob: {current_prob:.4f}")



In [ ]:
# 6. 学習結果の可視化
prob_after = get_vdb_routing_prob(model, test_x, test_y_in)
print(f"\n学習後の VectorDBSynapse の選択確率: {prob_after:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Probabilities
epochs_x = [i*10 for i in range(1, len(history_probs)+1)]
ax1.plot(epochs_x, history_probs, marker='o', color='blue', label='VectorDB Routing Prob')
ax1.set_title("Routing Probability to VectorDBSynapse")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Probability")
ax1.set_ylim(0, 1.05)
ax1.grid(True)
ax1.legend()

# Loss
ax2.plot(epochs_x, history_loss, marker='x', color='red', label='Total Loss')
ax2.set_title("Training Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.show()


## 考察
実験結果から、ファインチューニングを通じてルーターの Embedding が更新され、特定のタスク（入力特徴量）に対して自律的に `VectorDBSynapse` へトラフィックを寄せていく様子が確認できました。

- **初期状態**: ランダムな Embedding のため、選択確率は低い。
- **学習後**: わずかなステップ（数 epoch）の学習で、特定の入力に対して確信度高く VectorDB にルーティングできるようになります。

これにより、メタデータによる Hard Routing が難しい場合でも、少量の教師データさえあれば「いつどのカスタムシナプスを使うべきか」をモデルに教え込めることが証明されました。
